In [ ]:
%xmode Context
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt

import sys
from pathlib import Path

# help locating the package sgpykit (comment this out if you installed it)
PKG_PARENT = Path().resolve().parent
sys.path.insert(0, str(PKG_PARENT))

import sgpykit as sg

# Set up logging with sgpykit convenience functions
sg.set_logger_basic_format()
sg.set_logger_info_level()
# For more verbose output, use: sg.set_logger_debug_level()
# For custom format, use: sg.set_logger_custom_format(format)

# PART 5: Compute the g-PCE of a Function Given its Sparse Grid Approximation

The kit provides a function to compute the generalized Polynomial Chaos Expansion (g-PCE) of a function
of several variables, i.e. the expansion of $f$ in terms of a sum of orthonormal polynomials.

| Supported Random Variable | Orthonormal Polynomials      |
|---------------------------|------------------------------|
| Uniform                   | Legendre, Chebyshev          |
| Normal                    | Hermite                      |
| Exponential               | Laguerre                     |
| Gamma                     | Generalized Laguerre         |
| Beta                      | Jacobi                       |

The coefficients of these expansions are defined as suitable integrals over the space of parameters, and
could thus be approximated with sparse grid quadrature. However, a more efficient technique can be
applied, and it is actually implemented in the Kit. It consists in rearranging the sparse grid
interpolant, which is a linear combination of Lagrange polynomials, as a summation of orthonormal
polynomials (i.e. performing a change of basis to express the same polynomial). Given the relations
between sparse grids and orthogonal expansion, it is always possible to tune the sparse grid so to
obtain the gPCE in a precise polynomial space.

See e.g. Back Nobile Tamellini Tempone, `Stochastic Spectral Galerkin and Collocation...a  numerical
comparison'' for more details on the sparse grid/orthogonal expansion relation and Tamellini ph.D.
thesis, chap.6 or MOX report 13/2012 by Formaggia Guadagnini Imperiali Lever Porta Riva Scotti Tamellini
for details on the conversions.

More examples with different kinds of random variables / orthogonal polynomials can be found in test_convert_to_modal.m.

In [ ]:
# the sparse grid
N = 2
w = 5
knots = lambda n: sg.knots_uniform(n,-1,1,'nonprob')
lev2knots = sg.lev2knots_lin
idxset = lambda i: np.prod(i+1, axis=0)

S,_ = sg.create_sparse_grid(N, w, knots, lev2knots, idxset)
Sr = sg.reduce_sparse_grid(S)

# the domain of the grid
domain = np.vstack((-np.ones(N), np.ones(N)))

# compute a legendre polynomial over the sparse grid
X = Sr.knots
nodal_values = 4*sg.lege_eval_multidim(X,[4, 0],-1,1)+ 2*sg.lege_eval_multidim(X,[1, 1],-1,1)

# conversion from the points to the legendre polynomial. I should recover it exactly
modal_coeffs,K = sg.convert_to_modal(S, Sr, nodal_values, domain, 'legendre')

np.hstack((K, modal_coeffs))

# PART 6: Sparse-Grids-Based Sensitivity Analysis

## Compute Sobol Indices of a Function

In [ ]:
import numpy as np

# Define the domain
aa = np.array([-1, -1, -1])
bb = np.array([1, 1, 1])

# Define the functions
def f1(x):
    x = np.atleast_2d(x)
    return 1 + x[0, :]**2 + x[1, :]**2 + x[2, :]**2

def f2(x):
    x = np.atleast_2d(x)
    return 1 + 5 * x[0, :]**2 + x[1, :]**2 + x[2, :]**2

def f3(x):
    x = np.atleast_2d(x)
    return 1 / (1 + x[0, :]**2 + x[1, :]**2 + x[2, :]**2)

def f4(x):
    x = np.atleast_2d(x)
    return 1 / (1 + 5 * x[0, :]**2 + x[1, :]**2 + x[2, :]**2)

def f(x):
    return np.vstack((f1(x), f2(x), f3(x), f4(x)))

# We expect to see these results:
#   f1: has no mixed effects, so the principal and total Sobol indices are identical. Also, it's isotropic, so the indices of each variable are identical
#   f2: no mixed effects as f1, but y_1 contributes more to the variability of f so it has a larger Sobol total/principal index
#   f3: this function has mixed effects (partial derivatives are nonzero),  so the principal and total Sobol index will be different, but equal among random variables
#   f4: mixed effects, and y_1 contributes more to the variability of f so it has larger Sobol indices

# Generate a sparse grid
domain = np.vstack((aa, bb))
knots = lambda n: sg.knots_CC(n, -1, 1, 'nonprob')
N = len(aa)
w = 5
S,_ = sg.create_sparse_grid(N, w, knots, sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

values_on_grid,*_ = sg.evaluate_on_sparse_grid(f, S=None, Sr=Sr)

# Compute Sobol indices
Sob_i1, Tot_Sob_i1, Mean1, Var1 = sg.compute_sobol_indices_from_sparse_grid(S, Sr, values_on_grid[0, :], domain, 'legendre')
Sob_i2, Tot_Sob_i2, Mean2, Var2 = sg.compute_sobol_indices_from_sparse_grid(S, Sr, values_on_grid[1, :], domain, 'legendre')
Sob_i3, Tot_Sob_i3, Mean3, Var3 = sg.compute_sobol_indices_from_sparse_grid(S, Sr, values_on_grid[2, :], domain, 'legendre')
Sob_i4, Tot_Sob_i4, Mean4, Var4 = sg.compute_sobol_indices_from_sparse_grid(S, Sr, values_on_grid[3, :], domain, 'legendre')

# Display results
print('      f1   |    f2    |   f3    |    f4   ')
print('Principal Sobol indices')
print(np.column_stack((Sob_i1, Sob_i2, Sob_i3, Sob_i4)))
print('Total Sobol indices')
print(np.column_stack((Tot_Sob_i1, Tot_Sob_i2, Tot_Sob_i3, Tot_Sob_i4)))

## Compute Gradients of a Sparse Grid Interpolant (by Finite Differences)

In [ ]:
# define sparse grid over [4,6] x [1,5]
N=2
aa=np.array([4, 1])
bb=np.array([6, 5])

# the function to be interpolated and its derivatives
def f(x):
    y = 1./(1+0.5*sum(x**2))
    return y
    #return y[0]

def df1(x):
    y = -1./((1+0.5*sum(x**2))**2)*2*0.5*x[0,:]
    return y

def df2(x):
    y = -1./((1+0.5*sum(x**2))**2)*2*0.5*x[1,:]
    return y


# create a sparse grid and evaluate the function on it
domain = np.vstack((aa, bb))
knots1 = lambda n: sg.knots_CC(n,aa[0],bb[0],'nonprob')
knots2 = lambda n: sg.knots_CC(n,aa[1],bb[1],'nonprob')
w = 4
S,_ = sg.create_sparse_grid(N,w,[knots1,knots2],sg.lev2knots_doubling);
Sr = sg.reduce_sparse_grid(S)

values_on_grid,*_=sg.evaluate_on_sparse_grid(f,S=None,Sr=Sr)

# generate M random points in the domain where we evaluate the derivative of the sparse grid
# and the true derivative, to check error
M=100
# use get interval map to go from [-1,1]^N to actual domain
my_map=sg.get_interval_map(aa,bb,'uniform')

rng = np.random.default_rng(seed=42)
eval_points = my_map(rng.random(size=(N,M))*2-1)


# compute values with function
Grads = sg.derive_sparse_grid(S,Sr,values_on_grid,domain,eval_points)


# error and visualization

print(max(abs(Grads[0,:] - df1(eval_points))))
print(max(abs(Grads[1,:] - df2(eval_points))))

In [ ]:
fig, axs = sg.figure_create(nrows=1, ncols=2, figsize=(8, 4))
sg.plot(axs[0], Grads[0,:],'-o','DisplayName','Finite Diff')
sg.plot(axs[0], df1(eval_points),'-','DisplayName','true val')
axs[0].legend()

sg.plot(axs[1], Grads[1,:],'-o','DisplayName','Finite Diff')
sg.plot(axs[1], df2(eval_points),'-','DisplayName','true val')
axs[1].legend()

fig.tight_layout() 

In [ ]:
# define sparse grid over [4,6] x [1,5]
N=2
aa=np.array([4, 1])
bb=np.array([6, 5])

# the function to be interpolated and its derivatives
def f(x):
    y = 1./(1+0.5*sum(x**2))
    return y
    #return y[0]

def df1(x):
    y = -1./((1+0.5*sum(x**2))**2)*2*0.5*x[0,:]
    return y

def df2(x):
    y = -1./((1+0.5*sum(x**2))**2)*2*0.5*x[1,:]
    return y


# create a sparse grid and evaluate the function on it
domain = np.vstack((aa, bb))
knots1 = lambda n: sg.knots_CC(n,aa[0],bb[0],'nonprob')
knots2 = lambda n: sg.knots_CC(n,aa[1],bb[1],'nonprob')
w = 4
S,_ = sg.create_sparse_grid(N,w,[knots1,knots2],sg.lev2knots_doubling);
Sr = sg.reduce_sparse_grid(S)

values_on_grid,*_=sg.evaluate_on_sparse_grid(f,S=None,Sr=Sr)

# generate M random points in the domain where we evaluate the derivative of the sparse grid
# and the true derivative, to check error
M=100
# use get interval map to go from [-1,1]^N to actual domain
my_map=sg.get_interval_map(aa,bb,'uniform')

rng = np.random.default_rng(seed=42)
eval_points = my_map(rng.random(size=(N,M))*2-1)


# compute values with function
Grads = sg.derive_sparse_grid(S,Sr,values_on_grid,domain,eval_points)


# error and visualization

print(max(abs(Grads[0,:] - df1(eval_points))))
print(max(abs(Grads[1,:] - df2(eval_points))))

In [ ]:
# h is computed automatically in each direction as (b-a)/1E5, but can be adjusted if needed.
# In the example below, the length of the interval along direction 1 is O(1E-5) so choosing
# the default h would lead to h = O(1E-10), which incurs in numerical cancellations.
# Thus, setting manually a larger value for h helps in reducing the error

N=2

aa=np.array([4E-5, 1])
bb=np.array([6E-5, 5])

# the function to be interpolated and its derivatives
# ... same as above

# create a sparse grid and evaluate the function on it
domain = np.vstack((aa, bb))
knots1 = lambda n: sg.knots_CC(n,aa[0],bb[0],'nonprob')
knots2 = lambda n: sg.knots_CC(n,aa[1],bb[1],'nonprob')
w = 5; #
S,_ = sg.create_sparse_grid(N,w,[knots1,knots2],sg.lev2knots_doubling);
Sr = sg.reduce_sparse_grid(S)


values_on_grid,*_=sg.evaluate_on_sparse_grid(f,S=None,Sr=Sr)

# generate M random points in the domain where we evaluate the derivative of the sparse grid
# and the true derivative, to check error
M=100
# use get interval map to go from [-1,1]^N to actual domain
my_map=sg.get_interval_map(aa,bb,'uniform')

rng = np.random.default_rng(seed=42)
eval_points = my_map(rng.random(size=(N,M))*2-1)

# compute values with function
Grads_def = sg.derive_sparse_grid(S,Sr,values_on_grid,domain,eval_points)
h=np.array([1E-7, 1E-5])
Grads_man = sg.derive_sparse_grid(S,Sr,values_on_grid,domain,eval_points,h)

In [ ]:
# error and visualization
fig, axs = sg.figure_create(nrows=1, ncols=2, figsize=(8, 4))
sg.plot(axs[0], Grads_def[0,:],'-o','DisplayName','Finite Diff, default h')
sg.plot(axs[0], Grads_man[0,:],'x','DisplayName','Finite Diff, manual h')
sg.plot(axs[0], df1(eval_points),'-','DisplayName','true val')
axs[0].legend()

sg.plot(axs[1], Grads_def[1,:],'-o','DisplayName','Finite Diff, default h')
sg.plot(axs[1], Grads_man[1,:],'x','DisplayName','Finite Diff, manual h')
sg.plot(axs[1], df2(eval_points),'-','DisplayName','true val')
axs[1].legend()

fig.tight_layout()

In [ ]:
# Error
print(max(abs((Grads_def[0,:] - df1(eval_points))/df1(eval_points))))  # matlab: 1.9397
print(max(abs((Grads_man[0,:] - df1(eval_points))/df1(eval_points))))  # matlab: 0.0046492


## A function to compute Hessians of a function (by finite differences) is also available; see hessian_sparse_grid().

# PART 7: Save Sparse Grid on File

In [ ]:
N = 3

aa = [4, 1, -2]
bb = [6, 5, -1]

knots1 = lambda n: sg.knots_CC(n, aa[0], bb[0], 'nonprob')
knots2 = lambda n: sg.knots_CC(n, aa[1], bb[1], 'nonprob')
knots3 = lambda n: sg.knots_uniform(n, aa[2], bb[2], 'nonprob')

w = 2
S,_ = sg.create_sparse_grid(N, w, [knots1, knots2, knots3], sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

# Save points to 'points.dat'. The first row actually contains two integer
# values, i.e., Sr.size and N
sg.export_sparse_grid_to_file(Sr)

# Save points to 'mygrid.dat'
sg.export_sparse_grid_to_file(Sr, 'mygrid2.dat')

# Save points and weights to 'mygrid_with_weights.dat'
sg.export_sparse_grid_to_file(Sr, 'mygrid_with_weights2.dat', with_weights=True)